In [7]:
import fitz
import os

In [8]:
def extract_pdf_text(file_path):
    text = ""
    
    pdf = fitz.open(file_path)
    
    for page in pdf:
        text += page.get_text()
    
    return text

In [9]:
file_path = "../data/raw/nda_1.pdf"

contract_text = extract_pdf_text(file_path)

print(contract_text[:2000])

Mutual NDA – revised 4 May 2018
1
Barclays Commercial, Innovation & Technology Legal
______________________________________
NON-DISCLOSURE AGREEMENT
______________________________________
Note: This document is a Non-Disclosure Agreement only. A separate contract will be required
whenever:
-
goods or services are being provided by or to Barclays;
-
any payment is being made by or to Barclays;
-
any personal data is being transferred by or to Barclays; or
-
any intellectual property is being created or licensed by either party to the other (including
any rights to use Barclays’ name or logo).
Barclays:
Name and company registration
number:
(choose 
the 
applicable
Barclays contracting entity)
Registered address:
1 Churchill Place, London, E14 5HP
Counterparty:
Name:
(Full corporate or personal name of
other party)
Registered
address:
(or 
home
address where individual)
Country of registration: (leave
blank where counterparty is an
individual)
Company registration number:
(leave blank wh

In [10]:
output_path = "../data/processed/nda_1.txt"

with open(output_path, "w", encoding="utf-8") as file:
    file.write(contract_text)

print("Text saved successfully")

Text saved successfully


In [11]:
input_folder = "../data/raw/"
output_folder = "../data/processed/"

for filename in os.listdir(input_folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(input_folder, filename)

        text = extract_pdf_text(file_path)

        output_path = os.path.join(
            output_folder,
            filename.replace(".pdf", ".txt")
        )

        with open(output_path, "w", encoding="utf-8") as file:
            file.write(text)

print("All contracts extracted successfully")

All contracts extracted successfully


In [12]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = " ".join(text.split())
    return text

In [13]:
cleaned_text = clean_text(contract_text)

print(cleaned_text[:2000])

Mutual NDA – revised 4 May 2018 1 Barclays Commercial, Innovation & Technology Legal ______________________________________ NON-DISCLOSURE AGREEMENT ______________________________________ Note: This document is a Non-Disclosure Agreement only. A separate contract will be required whenever: - goods or services are being provided by or to Barclays; - any payment is being made by or to Barclays; - any personal data is being transferred by or to Barclays; or - any intellectual property is being created or licensed by either party to the other (including any rights to use Barclays’ name or logo). Barclays: Name and company registration number: (choose the applicable Barclays contracting entity) Registered address: 1 Churchill Place, London, E14 5HP Counterparty: Name: (Full corporate or personal name of other party) Registered address: (or home address where individual) Country of registration: (leave blank where counterparty is an individual) Company registration number: (leave blank where

In [14]:
cleaned_output_path = "../data/processed/cleaned_nda_1.txt"

with open(cleaned_output_path, "w", encoding="utf-8") as file:
    file.write(cleaned_text)

print("Cleaned text saved successfully")

Cleaned text saved successfully


In [15]:
import re

def split_clauses(text):
    clauses = re.split(r'\.\s+', text)

    return clauses

In [16]:
clauses = split_clauses(cleaned_text)

print(clauses[:10])

['Mutual NDA – revised 4 May 2018 1 Barclays Commercial, Innovation & Technology Legal ______________________________________ NON-DISCLOSURE AGREEMENT ______________________________________ Note: This document is a Non-Disclosure Agreement only', 'A separate contract will be required whenever: - goods or services are being provided by or to Barclays; - any payment is being made by or to Barclays; - any personal data is being transferred by or to Barclays; or - any intellectual property is being created or licensed by either party to the other (including any rights to use Barclays’ name or logo)', 'Barclays: Name and company registration number: (choose the applicable Barclays contracting entity) Registered address: 1 Churchill Place, London, E14 5HP Counterparty: Name: (Full corporate or personal name of other party) Registered address: (or home address where individual) Country of registration: (leave blank where counterparty is an individual) Company registration number: (leave blank

In [17]:
print("Total clauses:", len(clauses))

Total clauses: 51


In [18]:
import pandas as pd
df = pd.DataFrame({"clause": clauses})
df.to_csv("../data/datasets/clauses.csv", index=False)
print("Clauses saved successfully")

Clauses saved successfully


In [19]:
df.head(20)

,clause
0,Mutual NDA – revised 4 May 2018 1 Barclays Com...
1,A separate contract will be required whenever:...
2,Barclays: Name and company registration number...
3,“Evaluations by Barclays of the Counterparty’s...
4,The parties have agreed that any dealings betw...
5,Executed by the parties or their respective du...
6,For and on behalf of Barclays: For and on beha...
7,The terms Controlling and Controlled shall hav...
8,2 Confidential Information 2.1 The Receiving P...
9,2.2 The Receiving Party shall only use the Con...


In [20]:
df["label"] = ""
df.head(20)

,clause,label
0,Mutual NDA – revised 4 May 2018 1 Barclays Com...,
1,A separate contract will be required whenever:...,
2,Barclays: Name and company registration number...,
3,“Evaluations by Barclays of the Counterparty’s...,
4,The parties have agreed that any dealings betw...,
5,Executed by the parties or their respective du...,
6,For and on behalf of Barclays: For and on beha...,
7,The terms Controlling and Controlled shall hav...,
8,2 Confidential Information 2.1 The Receiving P...,
9,2.2 The Receiving Party shall only use the Con...,


In [21]:
df.loc[0, "label"] = "Confidentiality"
df.loc[1, "label"] = "Termination"
df.loc[2, "label"] = "Payment"
df.loc[3, "label"] = "Liability"
df.loc[4, "label"] = "Other"

In [22]:
df.to_csv("../data/datasets/labeled_clauses.csv", index=False)

print("Labeled dataset saved")

Labeled dataset saved


In [23]:
df = df[df["clause"].str.strip() != ""]

In [24]:
df.head()

,clause,label
0,Mutual NDA – revised 4 May 2018 1 Barclays Com...,Confidentiality
1,A separate contract will be required whenever:...,Termination
2,Barclays: Name and company registration number...,Payment
3,“Evaluations by Barclays of the Counterparty’s...,Liability
4,The parties have agreed that any dealings betw...,Other


In [25]:
df = df[df["label"] != ""]

In [26]:
X = df["clause"]
y = df["label"]

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_vectorized = vectorizer.fit_transform(X)

In [28]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_vectorized, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [29]:
sample_clause = ["Payment must be completed within 30 days"]

sample_vector = vectorizer.transform(sample_clause)

prediction = model.predict(sample_vector)

print("Predicted Label:", prediction[0])

Predicted Label: Termination


In [30]:
df.loc[0, "label"] = "Confidentiality"
df.loc[1, "label"] = "Termination"
df.loc[2, "label"] = "Payment"
df.loc[3, "label"] = "Liability"
df.loc[4, "label"] = "Other"

df.loc[5, "label"] = "Confidentiality"
df.loc[6, "label"] = "Termination"
df.loc[7, "label"] = "Payment"
df.loc[8, "label"] = "Liability"
df.loc[9, "label"] = "Other"

df.loc[10, "label"] = "Confidentiality"
df.loc[11, "label"] = "Termination"
df.loc[12, "label"] = "Payment"
df.loc[13, "label"] = "Liability"
df.loc[14, "label"] = "Other"

df.loc[15, "label"] = "Confidentiality"
df.loc[16, "label"] = "Termination"
df.loc[17, "label"] = "Payment"
df.loc[18, "label"] = "Liability"
df.loc[19, "label"] = "Other"

print("Labels added successfully")

Labels added successfully


In [31]:
new_rows = pd.DataFrame({
    "clause": [
        "Payment must be made within 30 days",
        "The client shall pay the amount before due date",
        "Invoice payment is required within 15 days",
        "All payments shall be completed electronically",
        "Late payment charges will apply"
    ],
    "label": [
        "Payment",
        "Payment",
        "Payment",
        "Payment",
        "Payment"
    ]
})

df = pd.concat([df, new_rows], ignore_index=True)

print("Payment samples added")

Payment samples added


In [32]:
# Remove empty labels
df = df[df["label"] != ""]

# Remove empty clauses / NaN values
df = df.dropna(subset=["clause"])

# Convert clauses to string
df["clause"] = df["clause"].astype(str)

# Prepare training data
X = df["clause"]
y = df["label"]

# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_vectorized = vectorizer.fit_transform(X)

# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_vectorized, y)

# Test prediction
sample_clause = ["Payment must be completed within 30 days"]

sample_vector = vectorizer.transform(sample_clause)

prediction = model.predict(sample_vector)

print("Predicted Label:", prediction[0])

Predicted Label: Payment


In [33]:
import pickle

# Save model
with open("../models/clause_model.pkl", "wb") as file:
    pickle.dump(model, file)

# Save vectorizer
with open("../models/vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

print("Model and vectorizer saved successfully")

Model and vectorizer saved successfully


In [34]:
import pickle

# Load saved model
with open("../models/clause_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

# Load saved vectorizer
with open("../models/vectorizer.pkl", "rb") as file:
    loaded_vectorizer = pickle.load(file)

# Test prediction
sample_clause = ["Confidential information must not be shared"]

sample_vector = loaded_vectorizer.transform(sample_clause)

prediction = loaded_model.predict(sample_vector)

print("Predicted Label:", prediction[0])

Predicted Label: Payment


<h1> More Accurate Version <h1>

In [35]:
training_data = pd.DataFrame({
    "clause": [

        # Payment
        "Payment must be completed within 30 days",
        "Invoice payment is required within 15 days",
        "Late payment charges will apply",
        "The client shall pay all fees on time",
        "All payments shall be processed electronically",

        # Confidentiality
        "Confidential information must not be disclosed",
        "Both parties agree to maintain confidentiality",
        "The employee shall not share trade secrets",
        "Sensitive company data must remain private",
        "Disclosure of confidential material is prohibited",

        # Termination
        "Either party may terminate this agreement",
        "The contract may be terminated with notice",
        "Termination requires 30 days prior notice",
        "This agreement shall end upon breach",
        "The company reserves the right to terminate",

        # Liability
        "The company shall not be liable for damages",
        "Neither party accepts unlimited liability",
        "Liability is limited under this agreement",
        "The vendor assumes all legal responsibility",
        "Damages shall be compensated accordingly",

        # Other
        "This agreement is governed by applicable law",
        "The parties agree to the stated conditions",
        "This contract becomes effective immediately",
        "The agreement shall remain legally binding",
        "All amendments must be documented"
    ],

    "label": [

        "Payment",
        "Payment",
        "Payment",
        "Payment",
        "Payment",

        "Confidentiality",
        "Confidentiality",
        "Confidentiality",
        "Confidentiality",
        "Confidentiality",

        "Termination",
        "Termination",
        "Termination",
        "Termination",
        "Termination",

        "Liability",
        "Liability",
        "Liability",
        "Liability",
        "Liability",

        "Other",
        "Other",
        "Other",
        "Other",
        "Other"
    ]
})

df = pd.concat([df, training_data], ignore_index=True)

print("Training data added successfully")

Training data added successfully


In [36]:
# Remove missing values
df = df.dropna(subset=["clause", "label"])

# Convert to string
df["clause"] = df["clause"].astype(str)

# Prepare data
X = df["clause"]
y = df["label"]

# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_vectorized = vectorizer.fit_transform(X)

# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_vectorized, y)

print("Model retrained successfully")

Model retrained successfully


In [37]:
sample_clause = ["Confidential information must not be shared"]

sample_vector = vectorizer.transform(sample_clause)

prediction = model.predict(sample_vector)

print("Predicted Label:", prediction[0])

Predicted Label: Confidentiality


<h1>Risk Detection<h1>

In [38]:
risk_data = pd.DataFrame({"clause": [

        # High Risk
        "The company accepts unlimited liability",
        "The client may terminate immediately without notice",
        "All legal responsibility falls on the vendor",
        "The employee is liable for all damages",
        "Confidential information may be shared publicly",

        # Medium Risk
        "The agreement renews automatically every year",
        "Late payment penalties may apply",
        "Termination requires a short notice period",
        "Additional charges may be added later",
        "The service terms may change without warning",

        # Low Risk
        "Payment must be completed within 30 days",
        "Both parties agree to maintain confidentiality",
        "The contract may be terminated with prior notice",
        "All amendments must be documented",
        "The agreement follows standard company policy"
    ],

    "risk": [

        "High Risk",
        "High Risk",
        "High Risk",
        "High Risk",
        "High Risk",

        "Medium Risk",
        "Medium Risk",
        "Medium Risk",
        "Medium Risk",
        "Medium Risk",

        "Low Risk",
        "Low Risk",
        "Low Risk",
        "Low Risk",
        "Low Risk"
    ]
})

print(risk_data.head())

                                              clause       risk
0            The company accepts unlimited liability  High Risk
1  The client may terminate immediately without n...  High Risk
2       All legal responsibility falls on the vendor  High Risk
3             The employee is liable for all damages  High Risk
4    Confidential information may be shared publicly  High Risk


In [39]:
# Prepare risk training data
X_risk = risk_data["clause"]
y_risk = risk_data["risk"]

# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

risk_vectorizer = TfidfVectorizer()

X_risk_vectorized = risk_vectorizer.fit_transform(X_risk)

# Train model
from sklearn.linear_model import LogisticRegression

risk_model = LogisticRegression(max_iter=1000)

risk_model.fit(X_risk_vectorized, y_risk)

print("Risk Detection Model trained successfully")

Risk Detection Model trained successfully


In [40]:
sample_clause = ["The company accepts unlimited liability"]

sample_vector = risk_vectorizer.transform(sample_clause)

prediction = risk_model.predict(sample_vector)

print("Predicted Risk:", prediction[0])

Predicted Risk: High Risk


In [41]:
import pickle

# Save risk model
with open("../models/risk_model.pkl", "wb") as file:
    pickle.dump(risk_model, file)

# Save risk vectorizer
with open("../models/risk_vectorizer.pkl", "wb") as file:
    pickle.dump(risk_vectorizer, file)

print("Risk model and vectorizer saved successfully")

Risk model and vectorizer saved successfully


In [42]:
import pickle

# Load saved risk model
with open("../models/risk_model.pkl", "rb") as file:
    loaded_risk_model = pickle.load(file)

# Load saved risk vectorizer
with open("../models/risk_vectorizer.pkl", "rb") as file:
    loaded_risk_vectorizer = pickle.load(file)

# Test prediction
sample_clause = ["The employee is liable for all damages"]

sample_vector = loaded_risk_vectorizer.transform(sample_clause)

prediction = loaded_risk_model.predict(sample_vector)

print("Predicted Risk:", prediction[0])

Predicted Risk: High Risk


In [43]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully")

spaCy model loaded successfully


In [44]:
sample_text = """
ABC Pvt Ltd will pay ₹50000 to XYZ Solutions on January 10, 2026 in Mumbai.
"""

doc = nlp(sample_text)

for ent in doc.ents:
    print(ent.text, "→", ent.label_)

ABC Pvt Ltd → ORG
50000 → MONEY
XYZ Solutions → ORG
January 10, 2026 → DATE
Mumbai → GPE


In [1]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

print("Summarizer loaded successfully")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

C:\Users\stwks\anaconda3\envs\contract_ai\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\stwks\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Summarizer loaded successfully


In [2]:
sample_contract = """
This agreement is made between ABC Pvt Ltd and XYZ Solutions.
The client agrees to complete payment within 30 days.
Both parties agree to maintain confidentiality of sensitive information.
The agreement may be terminated with prior written notice.
The company shall not be liable for indirect damages.
"""

summary = summarizer(
    sample_contract,
    max_length=60,
    min_length=20,
    do_sample=False
)

print(summary[0]['summary_text'])

This agreement is made between ABC Pvt Ltd and XYZ Solutions. Both parties agree to maintain confidentiality of sensitive information.


In [47]:
def analyze_contract(text):

    # Clean text
    cleaned = clean_text(text)

    # Split clauses
    clauses = split_clauses(cleaned)

    print("\n--- Clause Classification ---")

    for clause in clauses[:5]:

        clause_vector = vectorizer.transform([clause])

        clause_prediction = model.predict(clause_vector)

        print(f"\nClause: {clause}")
        print("Type:", clause_prediction[0])

    print("\n--- Risk Detection ---")

    for clause in clauses[:5]:

        risk_vector = risk_vectorizer.transform([clause])

        risk_prediction = risk_model.predict(risk_vector)

        print(f"\nClause: {clause}")
        print("Risk:", risk_prediction[0])

    print("\n--- Named Entities ---")

    doc = nlp(text)

    for ent in doc.ents:
        print(ent.text, "→", ent.label_)

    print("\n--- Summary ---")

    summary = summarizer(
        text[:2000],
        max_length=60,
        min_length=20,
        do_sample=False
    )

    print(summary[0]['summary_text'])

In [48]:
analyze_contract(contract_text)


--- Clause Classification ---

Clause: Mutual NDA – revised 4 May 2018 1 Barclays Commercial, Innovation & Technology Legal ______________________________________ NON-DISCLOSURE AGREEMENT ______________________________________ Note: This document is a Non-Disclosure Agreement only
Type: Confidentiality

Clause: A separate contract will be required whenever: - goods or services are being provided by or to Barclays; - any payment is being made by or to Barclays; - any personal data is being transferred by or to Barclays; or - any intellectual property is being created or licensed by either party to the other (including any rights to use Barclays’ name or logo)
Type: Termination

Clause: Barclays: Name and company registration number: (choose the applicable Barclays contracting entity) Registered address: 1 Churchill Place, London, E14 5HP Counterparty: Name: (Full corporate or personal name of other party) Registered address: (or home address where individual) Country of registration: (